In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from urllib.parse import urljoin

BASE_URL = "https://astronomycenter.net"

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0"
})

# 🔹 CLEAN TEXT (fix excel error)
def clean_text(text):
    if text is None:
        return text
    return re.sub(r'[\x00-\x1F\x7F]', '', text)

# 🔹 Extract Hijri dari URL
def extract_hijri_from_url(relative_url):
    filename = relative_url.split("/")[-1].replace(".html", "")

    prefix_match = re.findall(r'[a-z]+', filename)
    year_match = re.findall(r'\d+', filename)

    if not prefix_match or not year_match:
        return "unknown", "unknown"

    prefix = prefix_match[0]
    year = year_match[0]

    month_map = {
        "muh": "Muharram",
        "saf": "Safar",
        "raa": "Rabee' Al-Awwal",
        "rat": "Rabee' Al-Aakher",
        "jua": "Jumadal Al-Aula",
        "jut": "Jumadal Al-Aakherah",
        "raj": "Rajab",
        "sha": "Sha'ban",
        "ram": "Ramadan",
        "shw": "Shawwal",
        "kea": "Dhul Qeadah",
        "hej": "Dhul Hijjah"
    }

    hijri_month = month_map.get(prefix, prefix)
    hijri_year = int(f"14{year}")

    return hijri_month, hijri_year

# 🔹 Ambil semua link bulan
def get_month_links():
    url = f"{BASE_URL}/res.html?l=en"
    print("Index:", url)

    res = session.get(url)
    soup = BeautifulSoup(res.text, "html.parser")

    table = soup.find("table")

    links = []
    for a in table.find_all("a", href=True):
        href = a["href"]

        if href.startswith("icop/") and href.endswith(".html"):
            links.append(href)

    
    return list(set(links))

# 🔹 Scrape 1 halaman
def scrape_page(relative_url):
    full_url = urljoin(BASE_URL, relative_url) + "?l=en"
    print("Scraping:", full_url)

    try:
        res = session.get(full_url, timeout=10)
        res.raise_for_status()
    except Exception as e:
        print("❌ Error:", e)
        return []

    soup = BeautifulSoup(res.text, "html.parser")

    anchor = soup.find("a", {"name": "waxing"})
    if anchor is None:
        print("⏭️ Skip (no waxing):", full_url)
        return []

    hijri_month, hijri_year = extract_hijri_from_url(relative_url)

    records = []
    current_date = None
    current_country = None

    for sib in anchor.find_next_siblings():

        # stop di waning
        if sib.name == "a" and sib.get("name") == "waning":
            break

        if sib.name == "h2" and "required" in sib.get("class", []):
            current_date = sib.get_text(strip=True)
            continue

        if sib.name == "h2":
            current_country = sib.get_text(strip=True)
            continue

        if sib.name == "div" and "observ" in sib.get("class", []):

            if current_date is None or current_country is None:
                continue

            text = sib.get_text(" ", strip=True)
            lower = text.lower()

            # 🎯 visibility
            if "not seen" in lower:
                visible = 0
            elif "seen" in lower:
                visible = 1
            else:
                visible = None

            # 🌦️ weather
            if "hazy" in lower:
                weather = "hazy"
            elif "cloudy" in lower:
                weather = "cloudy"
            elif "clear" in lower:
                weather = "clear"
            else:
                weather = "unknown"

            records.append({
                "date": current_date,
                "country": current_country,
                "visible": visible,
                "weather": weather,
                "observ": text,
                "hijri_month": hijri_month,
                "hijri_year": hijri_year,
                "source": full_url
            })

    return records

# 🔹 MAIN PIPELINE
def scrape_all():
    links = get_month_links()
    print(f"Total pages: {len(links)}")

    all_records = []

    for link in links:
        data = scrape_page(link)
        all_records.extend(data)

        time.sleep(0.5)  # ⏳ delay

    return all_records

# 🔹 RUN
records = scrape_all()

print("Total rows:", len(records))

df = pd.DataFrame(records)

# cleaning (WAJIB sebelum excel)
for col in df.select_dtypes(include=["object"]).columns:
    df[col] = df[col].apply(clean_text)

# convert date
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# save
df.to_excel("Crescent_Moon_Visibility_Dataset_(ICOP_Derived).xlsx", index=False)

print("✅ Saved to Crescent_Moon_Visibility_Dataset_(ICOP_Derived).xlsx")

Index: https://astronomycenter.net/res.html?l=en
Total pages: 348
Scraping: https://astronomycenter.net/icop/jua34.html?l=en
Scraping: https://astronomycenter.net/icop/saf28.html?l=en
⏭️ Skip (no waxing): https://astronomycenter.net/icop/saf28.html?l=en
Scraping: https://astronomycenter.net/icop/hej31.html?l=en
Scraping: https://astronomycenter.net/icop/sha26.html?l=en
⏭️ Skip (no waxing): https://astronomycenter.net/icop/sha26.html?l=en
Scraping: https://astronomycenter.net/icop/jua33.html?l=en
Scraping: https://astronomycenter.net/icop/saf45.html?l=en
Scraping: https://astronomycenter.net/icop/ram21.html?l=en
⏭️ Skip (no waxing): https://astronomycenter.net/icop/ram21.html?l=en
Scraping: https://astronomycenter.net/icop/raj28.html?l=en
⏭️ Skip (no waxing): https://astronomycenter.net/icop/raj28.html?l=en
Scraping: https://astronomycenter.net/icop/muh31.html?l=en
Scraping: https://astronomycenter.net/icop/raa42.html?l=en
Scraping: https://astronomycenter.net/icop/shw31.html?l=en
Scrap

In [4]:
df.to_csv("Crescent_Moon_Visibility_Dataset_(ICOP_Derived).csv", index=False)